# Introduction to Workbench Workbook

This workbook describes the solutions to the problems offered in the [Introduction to Workbench kata](./IntroToWorkbench.ipynb).



In [ ]:
from psiqdk.workbench import QPU, Qubits, Qubrick
import numpy as np
from typing import Callable

## Problem 1: Instantiate a QPU

All we need to do for this is pass the input argument `num_qubits` to the `num_qubits` parameter of a `QPU`. This creates a QPU with an upper bound of `num_qubits` qubits.

In [ ]:
def instantiate_qpu(num_qubits: int):
    qpu = QPU(num_qubits=num_qubits) # Instantiate the QPU here
    return qpu

Correct!


## Problem 2: Instantiate a Qubit Register

All we need to do for this is pass the input argument `num_qubits` to the first parameter of the `Qubits` object, the `qubit_name` to the second parameter, and the `qpu` to the third parameter. This will create a `Qubits` object that is linked to the QPU with the given size and given name.

In [ ]:
def instantiate_qubits(qpu: QPU, num_qubits: int, qubit_name: str):
    qubits = Qubits(num_qubits, qubit_name, qpu) # Instantiate the Qubits object here
    return qubits

Correct!


## Problem 3: Apply Quantum Gates

Since we have been given a `Qubits` object, and can apply gates directly to the `Qubits` object, we perform the two gates in order. First, we call `x()` on the qubit register.  This applies the `x` gate. Then we do the same with `rx` but pass the given parameter to apply the specific angle.

In [ ]:
def apply_gates(reg: Qubits, value: int):
    reg.x() # Apply an x gate to the qubit
    reg.rx(value) # Apply an rx gate with the given value to the qubit

Correct!


## Problem 4: Apply Controlled Quantum Gates

Since the first two controlled gates use `reg2` as the target, we call `x` gates on `reg2` twice. However, in the first gate we use only use `reg1` as the condition parameter since $\ket{1}$ is the default condition. In the second we use `reg1==0` to condition on the $\ket{0}$ state of `reg1` instead.

In the third gate, we instead target `reg1` so call an `x` gate on `reg1`, but use `reg2` as our condition.

In [ ]:
def apply_controlled_gates(reg1: Qubits, reg2: Qubits):
    reg2.x(cond=reg1) # Apply an controlled-x gate where reg1 controls reg2 on |1>
    reg2.x(cond=reg1==0) # Apply an controlled-x gate where reg1 controls reg2 on |0>
    reg1.x(cond=reg2) # Apply an controlled-x gate where reg2 controls reg1 on |1>

Correct!


## Problem 5: Indexing and Slicing

For the first gate, we need to apply an `x` gate to only one qubit, the second qubit in the register.  We can simply do this by directly indexing the register with an index of 1.

For the second gate, we need to apply an `x` gate to multiple qubits from the register at once.  We exclude the zeroth-indexed qubit since we are ignoring the first qubit, but use a start qubit from the next even-indexed qubit at index 2. Then we use a step of 2 to capture every other even-indexed qubit in the register.

In [ ]:
def indexing_and_slicing(reg: Qubits):
    reg[1].x() # Apply an x gate to the second qubit in the register with a single operation
    reg[2::2].x() # Apply an x gate to every even qubits except the first with a single operation

Correct!


## Problem 6: Combining Qubits

Since every concatenation returns a new `Qubits` object, we simply need to use the `|` operator between the requested sets, then apply the corresponding gate to generate the correct results.

In [ ]:
def concatenate_registers(reg1: Qubits, reg2: Qubits, reg3: Qubits):
    (reg1 | reg3).x() # Apply an x gate to the concatenation of the first and third registers
    (reg2 | reg3).y()  # Apply an y gate to the concatenation of the second and third registers
    (reg1 | reg2 | reg3).z() # Apply an z gate to the concatenation of the first, second and third registers

Correct!


## Problem 7: Using the State Vector

In this problem examining each component of the state vector. We first get the state vector by using `qpu.pull_state()`.

As the state vector is an array, we iterate over the state vector, keeping track of the amplitude and current index. Whenever we see a value that is equal to the current maximum (which starts at a value of 0), we add it to an existing list. If we find a value that is higher, we clear the list, and add the current index. If the value is smaller than the current maximum, we do nothing.

Finally, we return the `equal_amplitudes` list from the function.

In [ ]:
def examine_state_vector(qpu: QPU) -> list[int]:
    equal_amplitudes = []
    # Get the state vector and determine the indices with the maximum amplitude
    state_vector = qpu.pull_state()
    max_val = 0
    for ind, val in enumerate(state_vector):
        if val > max_val:
            equal_amplitudes = [ind]
            max_val = val
        elif np.isclose(val, max_val):
            equal_amplitudes.append(ind)
    return equal_amplitudes

Correct!


## Problem 8: Applying and Using Measurements

We are given a function that, when run, constructs a circuit that prepares a certain states on a return `Qubits` object. We can use the `read()` method on this object to get a value from this prepared state. In this case, the prepared state is a superposition between the all $\ket{0}$ case and the $\ket{1}$ case.

We use a for loop to run the function 1000 times. On each iteration, we read the value of the `Qubits` object. Then, we check if it exists in the current dictionary. If it does not, we set a counter equal to 0. Then we add one to that counter for the value read.

The dictionary is then returned from the function.

In [ ]:
def apply_and_use_measurements(func: Callable[[], Qubits]) -> dict[int, int]:
    result_dict = {}

    # Create a loop that runs 1000 times, runs the function, measures the result, and updates the dictionary
    for i in range(1000):
        qubit = func()
        res = qubit.read()
        if res not in result_dict:
            result_dict[res] = 0
        result_dict[res] += 1
    
    return result_dict

Correct!


## Problem 9: Building Qubricks

For the GHZ qubrick, we only need to define the `_compute` method.  We are given a single `Qubits` register. Since we need to put one control qubit into superposition, we apply a Hadamard gate to the zeroth-indexed qubit.  Then to entangle it with the other qubits in the register, we iterate over the qubits at indices 1 to $n-1$. We apply an `x` gate controlled by the zeroth-indexed qubit.  This entangles each other qubit in the register with the value of the first qubit in the register.

In [ ]:
class GHZ(Qubrick):

    def _compute(self, reg: Qubits):
        #... # Implement the GHZ circuit
        reg[0].had()
        for r in reg[1:]:
            r.x(cond=reg[0])

Correct!


## Problem 10: Apply a Qubrick

To apply the `GHZ` qubrick, we first instantiate it with no arguments. The `GHZ` qubrick has no high level parameters to set.

Then, we use the `compute` method of the `GHZ` qubrick, passing the register we want to apply the `GHZ` qubrick to.

Then, we return the `QPU` from the function.

In [ ]:
def apply_qubrick(n: int) -> QPU:
    qpu = QPU(num_qubits=n) # Build your QPU
    reg = Qubits(n, "reg", qpu=qpu)
    ghz = GHZ() # Instantiate your Qubrick
    ghz.compute(reg) # Apply the Qubrick
    return qpu

Correct!
